# Pixl: train

Trains `EncoderPixl` from scratch on the four-object scenes of [`ocdbytes/pixl-data`](https://huggingface.co/datasets/ocdbytes/pixl-data) with `pixl.train`, the same code that `python -m pixl.train` and the Docker image run. The first run downloads and encodes the scenes (about 1 GB, cached afterwards).

- **Train:** run every cell with the kernel's working directory set to `notebooks/`. Each evaluation prints a line and saves `checkpoints/<RUN>/checkpoint.pt` and `samples-<step>.png` relative to the project root. With `QUICK = True`, the folder is `<RUN>-quick` and evaluation runs every 20 steps. The defaults, `QUICK = True` on `DEVICE = "cpu"`, are a light 40-step check that leaves the Mac usable: training on its GPU froze the terminal. Real runs belong on the GPU server (`DEVICE = "auto"`, `QUICK = False`, or the Docker image): one pass over the million scenes is 31,250 steps.
- **Stop and resume:** interrupt the training cell (■). Running it again resumes from the last saved evaluation, with its saved random states. Changes to saved settings or input fingerprints stop the resume: change `RUN` to train with other settings. The device and checkpoint directory may change.
- **Look at a run:** run the imports and Settings code cells, then skip to Results. Set `RUN` and `QUICK` to match the checkpoint you want to view, including one trained on a server: copy its run folder into the project's `checkpoints/`. `generate.ipynb` draws scenes from your own captions.

In [ ]:
import os

# Apple GPU allocation limit: 0.9x the recommended maximum. Above 0.6x, attempt garbage collection
# and more frequent command-buffer commits. Read when the GPU allocator is first initialised.
os.environ.setdefault("PYTORCH_MPS_HIGH_WATERMARK_RATIO", "0.9")
os.environ.setdefault("PYTORCH_MPS_LOW_WATERMARK_RATIO", "0.6")

from pathlib import Path

import pandas as pd
import torch
from IPython.display import Image, display
from matplotlib import pyplot as plt

from pixl.train import KINDS, Args, load_model, run_folder, train

## Settings

`uv run python -m pixl.train --help` explains the command-line settings. `QUICK` overrides warm-up to 10 steps and evaluation to every 20 steps. `CPU_THREADS` controls this notebook's CPU thread count.

In [ ]:
RUN = "scenes-four"
QUICK = True  # True: 40 steps on 2,000 scenes, in <RUN>-quick; False: the full run
DEVICE = "cpu"  # "auto": CUDA, then Apple's GPU, then the CPU
CPU_THREADS = 4
BATCH = 32
EPOCHS = 1  # passes over the train scenes
PEAK_LR, MIN_LR, WARMUP = 3e-4, 3e-5, 1000
EVAL_EVERY = 2000

if DEVICE == "cpu":
    # A few threads, and this kernel at the lowest priority until it restarts, so the Mac stays responsive.
    torch.set_num_threads(CPU_THREADS)
    os.nice(max(0, 19 - os.nice(0)))

ARGS = Args(run=RUN, quick=QUICK, device=DEVICE, batch=BATCH, epochs=EPOCHS, peak_lr=PEAK_LR, min_lr=MIN_LR, warmup=WARMUP,
            eval_every=EVAL_EVERY, checkpoints=Path("..").joinpath("checkpoints"))  # the project's checkpoints/
FOLDER = run_folder(ARGS)
CHECKPOINT = FOLDER.joinpath("checkpoint.pt")
print(f"{CHECKPOINT} {'exists: training resumes from it' if CHECKPOINT.exists() else 'is new'}")

## Train

One line per evaluation: `batch loss` is the mean training loss since the last one; then each kind of evaluation scene's loss and **position**, how often its caption beats the same caption with two objects' quadrants exchanged (chance 0.5). Full runs evaluate every fourth scene during training and every scene at the end; quick runs use every 50th scene throughout, including the final scores.

In [ ]:
train(ARGS)

## Results

The measures at each evaluation, and the final scores once the run has finished (subsampled in quick mode). `new pair` position well above 0.5 is evidence that the model reads positions; high `train` position with `new pair` near 0.5 suggests weaker generalisation to unseen object combinations.

In [ ]:
_, state = load_model(CHECKPOINT)
table = pd.DataFrame(state["log"]).set_index("step")
display(table.round(3))
figure, axes = plt.subplots(1, 2, figsize=(14, 3.5))
axes[0].plot(table.index, table["batch loss"], label="training batches")
for kind in KINDS:
    axes[0].plot(table.index, table[f"{kind} loss"], marker="o", label=kind)
    axes[1].plot(table.index, table[f"{kind} position"], marker="o", label=kind)
axes[1].axhline(0.5, color="grey", linestyle=":", label="chance")
for axis, title in zip(axes, ("loss per pixel", "position")):
    axis.set_title(title)
    axis.set_xlabel("step")
    axis.legend(fontsize=8)
plt.show()
if "final" in state:
    display(pd.DataFrame({kind: {m: state["final"][f"{kind} {m}"] for m in ("loss", "position")} for kind in KINDS}).round(3))

## Samples during training

The preview at the last `SHOW` evaluations: 4 train scenes and 4 scenes with an object pair never seen together, real on top and the model's sample of the same caption below.

In [ ]:
SHOW = 4
for path in sorted(FOLDER.glob("samples-*.png"))[-SHOW:]:
    print(path.name)
    display(Image(filename=path))